In [11]:
data = {
    "text": [
        "hello",
        "hi",
        "good morning",
        "bye",
        "see you later",
        "goodbye",
        "what is your name",
        "who are you",
        "tell me your name",
        "how can I return product",
        "return policy",
        "how to return order"
    ],

    "intent": [
        "greeting",
        "greeting",
        "greeting",
        "goodbye",
        "goodbye",
        "goodbye",
        "name",
        "name",
        "name",
        "return_policy",
        "return_policy",
        "return_policy"
    ]
}

In [16]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from transformers import BertTokenizer

import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.optim import AdamW

from transformers import BertForSequenceClassification

df = pd.DataFrame(data)

l_en = LabelEncoder()
df['label'] = l_en.fit_transform(df['intent'])

no_labels = len(l_en.classes_)

print(no_labels)

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

tokens = tokenizer(list(df['text']), padding=True, truncation=True, return_tensors="pt")

class ChatDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key:val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

ds = ChatDataset(tokens, df['label'].tolist())

model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=no_labels)

loader = DataLoader(ds, batch_size=5, shuffle=True)

opt = AdamW(model.parameters(), lr=4e-5)

model.train()

#Actual model training is happening here
for epoch in range(6):
    for batch in loader:
        opt.zero_grad()

        op = model(input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            labels=batch['labels'])

        loss = op.loss
        loss.backward()

        opt.step()
    print("Epoch losses:", loss.item())


#Now we need to predict intent
def intent_pred(text):
    ip = tokenizer(text, truncation=True, padding=True, return_tensors="pt")

    ops = model(**ip)
    raw_pred = ops.logits
    pred = torch.argmax(raw_pred).item()

    return l_en.inverse_transform([pred])[0]

4


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch losses: 1.6048027276992798
Epoch losses: 1.2508593797683716
Epoch losses: 0.8557294607162476
Epoch losses: 0.7136826515197754
Epoch losses: 0.8985093832015991
Epoch losses: 0.3777485489845276


In [17]:
responses = {
    
    "greeting": "Hello! How can I help you?",
    
    "goodbye": "Goodbye! Have a great day!",
    
    "name": "I am an AI chatbot built using NLP.",
    
    "return_policy": "You can return the product within 30 days."
}

In [ ]:
while True:
    u_ip = input("User: ")
    if u_ip == "exit":
        break
    if u_ip == "goodbye":
        break
    
    intent = intent_pred(u_ip)
    reply = responses.get(intent, "Sorry I couldn't get it")

    print("AI Bot: ", reply)